# Solar Active-Region Detection — Kaggle training run (MAX-QUALITY preset)

**One cell to run.** Sets everything up (repo, Python env, data download) and starts
training on the GPU with the best-quality model the T4 can run.

Before you run it, check the right-hand panel (**Settings**):

1. **Internet: ON** — required for downloading the solar data and installing packages.
   (If the switch is missing, verify your phone number in Kaggle account settings.)
2. **GPU: T4 x2 or P100** — the run uses the first GPU automatically.

This preset (tuned for the 30 GB RAM / ~57 GB disk / T4 runtime):

- `BASE_CHANNELS=48 DEEP_SUPERVISION=1` — the best-quality model (18M params +
  auxiliary decoder losses), comfortably within T4 VRAM (batch size auto-fits)
- `DOWNLOAD_WORKERS=16` — the big disk holds 16 in-flight ~570 MB frames
- `MAX_TOTAL_FRAMES=2000` — same dataset size as the Mac run; disk self-limits if needed
- `CPU_HEADROOM=0` — every core to training (no thermal limit in the datacenter)

Kaggle stops a session after ~12 hours. **That is fine**: run the same cell again in
the same notebook and it resumes exactly where it stopped. The latest model is copied
to /kaggle/output every 5 minutes — downloadable from the Output tab after each session.

In [ ]:
%%bashset -xexport HOME=/kaggle/workcd /kaggle/workif [ -d SOALR/.git ]; then    git -C SOALR pull -q || trueelse    git clone -q -b arena/01a04247-soalr-active-region-detection \        https://github.com/haydenCoder/SOALR-ACTIVE-REGION-DETECTION-MODEL-SUN-.git SOALR \    || { mkdir -p SOALR && wget -qO /tmp/repo.tgz \        https://codeload.github.com/haydenCoder/SOALR-ACTIVE-REGION-DETECTION-MODEL-SUN-/tar.gz/refs/heads/arena/01a04247-soalr-active-region-detection \        && tar xzf /tmp/repo.tgz -C SOALR --strip-components=1; }fiif [ ! -x SOALR/scripts/run_forever.sh ]; then    echo "STOP: repository download failed. In the notebook Settings, make sure Internet is ON, then re-run this cell."    exit 1ficd SOALR# Reuse Kaggle's preinstalled CUDA torch (saves a ~2 GB download):if [ ! -x .venv/bin/python ]; then    python3 -m venv --system-site-packages .venvfi.venv/bin/python -m pip install -q -r requirements.txt# Kaggle only lets you DOWNLOAD files from /kaggle/output (zip available in the# notebook's "Output" tab after each session ends). This watcher keeps a fresh# copy of the latest model + log there every 5 minutes, so whatever session# dies, the Output tab holds the newest model:mkdir -p /kaggle/output(    while :; do        sleep 300        cp -f /kaggle/work/solar_results/arpil/continuous/best.pt    /kaggle/output/ 2>/dev/null        cp -f /kaggle/work/solar_results/arpil/continuous/last.pt    /kaggle/output/ 2>/dev/null        cp -f /kaggle/work/solar_results/arpil/continuous/metrics.jsonl /kaggle/output/ 2>/dev/null        cp -f /kaggle/work/solar_results/arpil/run_forever.log      /kaggle/output/ 2>/dev/null        cp -f /kaggle/work/solar_results/arpil/STATUS.md            /kaggle/output/ 2>/dev/null    done) &SAVE_WATCHER=$!# MAX-QUALITY preset for the Kaggle T4 runtime (30 GB RAM / ~57 GB disk):#   BASE_CHANNELS=48 + DEEP_SUPERVISION=1  best-quality model (18M params +#                                          auxiliary decoder losses); the T4 has#                                          the VRAM for it (batch auto-fits)#   DOWNLOAD_WORKERS=16                    the big disk holds 16 in-flight#                                          ~570 MB frames (saturates the link)#   MAX_TOTAL_FRAMES=2000                  same dataset as the Mac run#   MIN_FREE_GB=4                          disk safety reserve (auto-pauses below it)#   CPU_HEADROOM=0                         every core to trainingCHANNELS="aia94 aia131 aia1600 aia171 aia193 aia211 aia304 aia335 hmi_m hmi_bx hmi_by hmi_bz hmi_v" \BASE_CHANNELS=48 DEEP_SUPERVISION=1 \MIN_FREE_GB=4 MAX_TOTAL_FRAMES=2000 FRAMES_PER_CYCLE=200 DOWNLOAD_WORKERS=16 \CPU_HEADROOM=0 TILES_PER_EPOCH=200 VAL_EPOCH=10 VAL_SUBSET=300 \bash scripts/run_forever.shkill "$SAVE_WATCHER" 2>/dev/null

## What you will see

- **Pre-flight** `3 ok` lines, then `Downloading 200 frames with 16 parallel workers ...`
- `[compile] torch.compile active (C-level graph engine)` — the fast compiler works on T4
- `[stream] epoch=... loss=...` — a real (finite) loss; first `val_dice` at epoch 10
- Checkpoints in `/kaggle/work/solar_results/arpil/continuous/`, mirrored to `/kaggle/output/`

## If the session dies (12-hour cap or a crash)

Run this same cell again — same notebook. Nothing is lost; it resumes downloads and
training from the last saved checkpoint.